# 🗂️ Notebook 2: S3 (Object Storage) — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **Bucket** — namespace + policy.
- **Object** — `(bucket, key, version_id)` → blob + user metadata.
- **Chunk/Shard** — piece of the blob on one storage node.

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from datetime import datetime
from pydantic import BaseModel, Field

class Bucket(BaseModel):
    name: str
    versioning: bool = False
    created_at: datetime

class ObjectMeta(BaseModel):
    bucket: str
    key: str
    version_id: str
    size: int = Field(ge=0)
    etag: str
    content_type: str = "application/octet-stream"
    is_delete_marker: bool = False
    created_at: datetime

from datetime import datetime, timezone
om = ObjectMeta(bucket="b", key="hello.txt", version_id="v1",
                size=11, etag="abc", created_at=datetime.now(timezone.utc))
print(om.model_dump_json(indent=2))

## HTTP APIs

| Method | Path | What |
|---|---|---|
| PUT | `/{bucket}` | Create bucket |
| PUT | `/{bucket}/{key}` | Upload object |
| GET | `/{bucket}/{key}?versionId=..` | Download |
| DELETE | `/{bucket}/{key}` | Delete (or create tombstone if versioned) |
| GET | `/{bucket}?list` | List objects |
| POST | `/{bucket}/{key}?uploads` | Start multipart upload |


## Quick demo

In [ ]:
# Toy in-memory S3 with versioning
from collections import defaultdict
import uuid, hashlib

objects = defaultdict(list)   # (bucket,key) → [ (version_id, data, deleted) ]

def put(bucket, key, data):
    vid = uuid.uuid4().hex[:8]
    objects[(bucket,key)].append((vid, data, False))
    return vid

def delete(bucket, key):
    vid = uuid.uuid4().hex[:8]
    objects[(bucket,key)].append((vid, None, True))   # tombstone
    return vid

def get(bucket, key, version_id=None):
    vs = objects[(bucket,key)]
    if version_id:
        for v,d,t in vs:
            if v == version_id: return None if t else d
        return None
    for v,d,t in reversed(vs):
        return None if t else d
    return None

v1 = put("b","x",b"hello")
v2 = put("b","x",b"HELLO")
delete("b","x")
print("latest:", get("b","x"))
print("v1:", get("b","x", v1))

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.